In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# (No edit) Imports
import os
import json

import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import FastGradientMethod
from art.attacks.evasion import SaliencyMapMethod
from art.attacks.evasion import CarliniL2Method
# from art.attacks.evasion import Carlin
from sklearn.preprocessing import MinMaxScaler

sys.path.append(str(Path.cwd().parents[2]))

import os

import torch
from utils.models import OBU
from utils.functions import get_windowed_data, load_model_checkpoint

from sklearn.metrics import precision_score, recall_score, f1_score

/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Inputs
checkpoint_file="../../../saved_models/RandomPos-cenFL-updated.ckpt"
data_file = "../../../data/RandomPos_0709.csv"
train_perc = 80

norm_trained = False
collapse_output = True # True if using JSMA (maybe CW????)T


In [12]:
# (No edit) Load models, define wrappers
model = load_model_checkpoint(checkpoint_file, gpu=False)

class SequenceCrossEntropy(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.CrossEntropyLoss()

    def forward(self, a, b):
        # ART passes one-hot labels; CrossEntropyLoss needs class indices
        if b.dim() == 3:
            b = b.argmax(dim=-1)  # (batch, seq_len, num_classes) → (batch, seq_len)
        return self.loss(
            a.permute(0, 2, 1),  # (batch, seq_len, C) → (batch, C, seq_len)
            b.long()
        )

class NormalizedCfCWrapper(nn.Module):
    def __init__(self, modena_model, data_min, data_max):
        super().__init__()
        self.modena_model = modena_model
        self.norm_trained = norm_trained
        self.collapse_output = collapse_output
        if not self.norm_trained:
            # MUST wrap in torch.tensor() explicitly — do not assign raw numpy arrays
            self.register_buffer("data_min", torch.tensor(data_min, dtype=torch.float32))
            self.register_buffer("data_max", torch.tensor(data_max, dtype=torch.float32))

    def forward(self, x_normalized):        
        if not self.norm_trained:
            x_raw = x_normalized * (self.data_max - self.data_min) + self.data_min
        else:
            x_raw = x_normalized
        logits, _ = self.modena_model(x_raw)

        if self.collapse_output:
            return logits.mean(dim=1)
        else:
            return logits

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [25]:
# (No edit) Load data for og model (very time consuming part)
(out_x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, 
                                                                      normalize=norm_trained, 
                                                                      train_perc=train_perc)



In [31]:
# (no edit) FUNCTION to set model variables

wrapped_model = NormalizedCfCWrapper(modena_model=model.model,
                                    data_min=out_x_train.amin(dim=(0,1)), # x_train.min(axis=(0,1)) in numpy
                                    data_max=out_x_train.amax(dim=(0,1)))
criterion = SequenceCrossEntropy()
optimizer = optim.Adam(
    wrapped_model.parameters(),
    lr=0.001
)

classifier = PyTorchClassifier(
    model=wrapped_model,
    loss=criterion,
    optimizer=optimizer,
    input_shape=(10, 7),
    nb_classes=4, # the range [0, 3]; WRONG: the number of unique classes in y_test, len(np.unique(y_test.numpy()))
    clip_values=(0.0, 1.0), # for normalized
    device_type="cpu"
)

/tmp/ipykernel_783416/2165918670.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("data_min", torch.tensor(data_min, dtype=torch.float32))
/tmp/ipykernel_783416/2165918670.py:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("data_max", torch.tensor(data_max, dtype=torch.float32))


In [ ]:
# TEST: Test the model - ensure there's a high accuracy with the original model
print("TESTING |", f"norm_trained: {norm_trained} | collapse output {collapse_output}", "| NO wrapper")
out = model.test(out_x_train, y_test, mathy=True)

# # Test how replicable the results are
# not_same = 0
# for i in range(20):
#     print(f"=== {i+1}/20 ===")
#     out_test = model.test(x_test, y_test, mathy=True)
#     if out != out_test:
#         print("NOT SAME!")
#         not_same +=1
# print("# of times with distinct output:", not_same)

print(f"Out: {out}")

TESTING | norm_trained: False | collapse output True | NO wrapper
torch.Size([124774, 10, 20])
0.9643128342104006
0.9797491232802805
Model got 1226792/1247740 right.
Accuracy: 0.9832112459326462, Precision: 0.9643128342104006, Recall: 0.9797491232802805, F1 Score: 0.9719696949422882
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Out: (0.9719696949422882, 0.9797491232802805, 0.9643128342104006, 0.9832112459326462)


In [27]:
# IF NEEDED (if og model is was not norm_trained)- Load data at *normalized* 
if not norm_trained:
    print("Setting normalized data")
    (x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, 
                                                                        normalize=True, 
                                                                        train_perc=train_perc)


Setting normalized data


In [32]:
# TEST: Benign test

benign_y_test = y_test.numpy()
if collapse_output:
    y_test_collapsed = y_test[:, 0].numpy()  # just take first timestep, since all are the same
    benign_y_test = y_test_collapsed

benign_predictions = classifier.predict(x_test, batch_size=64)
benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
true_flat = benign_y_test.flatten()
pred_flat = benign_pred_classes.flatten()

accuracy = np.sum(benign_pred_classes == benign_y_test) / benign_pred_classes.size
precision =  precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

print(f"Benign accuracy: {accuracy}")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Benign accuracy: 0.9961931171558177
Precision: 0.9961990861326315
Recall: 0.9961931171558177
F1: 0.9961883773969892


In [29]:
# Adversarial test function
def adv_test(end_index : int, path : str, filename : str, Attack, **kwargs):
    print(f"=== kwargs: {kwargs} ===")

    adv_y_test = y_test.numpy()
    if collapse_output:
        y_test_collapsed = y_test[:, 0].numpy()  # just take first timestep, since all are the same
        adv_y_test = y_test_collapsed
    adv_y_test = adv_y_test[:end_index]
    attack = Attack(classifier, **kwargs)
    
    x_test_adv = attack.generate(x=x_test.numpy()[:end_index])#, y=y_test[:fraction])
    adversarial_predictions = classifier.predict(x_test_adv)
    pred_classes = np.argmax(adversarial_predictions, axis=-1)  # (N, seq_len)
    adversarial_accuracy = np.sum(pred_classes == adv_y_test) / pred_classes.size
    print(f"Adversarial accuracy: {adversarial_accuracy}")

    true_flat = adv_y_test.flatten()
    pred_flat = pred_classes.flatten()
    precision = precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
    recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

    # Save metrics to JSON in ./output
    os.makedirs(path, exist_ok=True)
    metrics = {
        "endIndex": end_index,
        "kwargs": kwargs,
        "accuracy": float(adversarial_accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    # output_path = f"{path}/adv_test_eps_{eps}.json"
    output_path = f"{path}/{filename}"
    with open(output_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Saved metrics to {output_path}")

    return metrics

In [33]:
# FGSM - eps = 0.1
# PGD - eps = 0.1, 
# JSMA - swept gamma | collapsed_output = true, theta = amount perturbed, gamma = fraction of features perturbed

adv_test(end_index = 5000, # len(y_test.numpy()),
        path = "../adv_data/denorm-cw",
        filename=f"adv.json",
        Attack=CarliniL2Method)

CarliniL2Method()


=== kwargs: {} ===


C&W L_2: 100%|██████████| 5000/5000 [1:41:58<00:00,  1.22s/it] 


Adversarial accuracy: 0.9554
Precision: 0.957634214760849
Recall: 0.9554
F1: 0.9557423970602369
Saved metrics to ../adv_data/denorm-cw/adv.json


{'endIndex': 5000,
 'kwargs': {},
 'accuracy': 0.9554,
 'precision': 0.957634214760849,
 'recall': 0.9554,
 'f1': 0.9557423970602369}

In [ ]:
# Not required - y_test on its output
nb_classes = (np.unique(y_test.numpy()))
nb_classes
# freq
zeros = 0
threes = 0
for row in y_test.numpy():
    if np.unique(row)[0] == 0:
        zeros += 1
    elif np.unique(row)[0] == 3:
        threes += 1
    else:
        print("EXCEPTION!")
print("0", zeros, "| 3", threes)

In [ ]:
# FGSM - eps = 0.1
# PGD - eps = 0.1, 
# JSMA - swept gamma | collapsed_output = true, theta = amount perturbed, gamma = fraction of features perturbed

for i in range(4, 11, 1):
        gamma = float(i/10)
        adv_test(end_index = 5000, # len(y_test.numpy()),
                path = "../adv_data/denorm-cw",
                filename=f"adv_gamma-{gamma}_end-5000.json",
                Attack=CarliniL2Method)

# CarliniL2Method()
"""
class SaliencyMapMethod(
    classifier: CLASSIFIER_CLASS_LOSS_GRADIENTS_TYPE,
    theta: float = 0.1,
    gamma: float = 1,
    batch_size: int = 1,
    verbose: bool = True
)

Parameters
classifier
A trained classifier.

theta
Amount of Perturbation introduced to each modified feature per step (can be positive or negative).

gamma
Maximum fraction of features being perturbed (between 0 and 1).

batch_size
Size of the batch on which adversarial samples are generated.

verbose
Show progress bars.

"""
# # epsilon sweep for PGD
# for i in range(3, 31, 1):
#     eps = float(i/100)
#     adv_test(eps = eps, 
#              path = "../adv_test/norm-pgd_max-iter-5",
#              max_iter=5)
# gamma = 0.3, ~33 min 45 sec
# gamma = 0.4


In [ ]:
# FGSM - eps = 0.1
# PGD - eps = 0.1, 
# JSMA - collapsed_output = true, theta = amount perturbed, gamma = fraction of features perturbed

for i in range(4, 11, 1):
        gamma = float(i/10)
        adv_test(end_index = 5000, # len(y_test.numpy()),
                path = "../adv_data/denorm-jsma",
                filename=f"adv_gamma-{gamma}_end-5000.json",
                Attack=SaliencyMapMethod,
                gamma=gamma)
SaliencyMapMethod()
# # epsilon sweep for PGD
# for i in range(3, 31, 1):
#     eps = float(i/100)
#     adv_test(eps = eps, 
#              path = "../adv_test/norm-pgd_max-iter-5",
#              max_iter=5)
# gamma = 0.3, ~33 min 45 sec
# gamma = 0.4


In [ ]:
# # Test if scores are replicable
# not_same = 0
# for i in range(20):
#     print(f"=== {i+1}/20 ===")
#     benign_predictions = classifier.predict(x_test, batch_size=64)
#     benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
#     true_flat = y_test.flatten()
#     pred_flat = benign_pred_classes.flatten()

#     accuracy_test = np.sum(benign_pred_classes == y_test.numpy()) / benign_pred_classes.size
#     precision_test =  precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
#     recall_test = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
#     f1_test = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

#     if accuracy == accuracy_test and precision == precision_test \
#         and recall == recall_test and f1 == f1_test:
#         pass
#     else:
#         print("NOT SAME! f1:", f1)

# print("# of distinct values:", not_same)

In [ ]:
from glob import glob
glob("../adv_data/*")